# Run Variant Pipeline

| Variant | NLI_REORDER | NLI_HINT |
|---|---|---|
| `a0` | 0 | 0 |
| `a1` | 1 | 0 |
| `a2` | 0 | 1 |
| `v_new` | 1 | 1 |

## Config

In [1]:
VARIANT_ID = 'a0'
REPORT_IDS = [
    'Bangchak2024', 'Bangchak2025', 'Bapco2023', 'Bapco2024', 'Bunduq2023',
    'Desfa2023', 'Energean2023', 'Energean2024', 'HPCL2023', 'HPCL2024',
    'IslandOil2024', 'Mubadala2023', 'Mubadala2024', 'OKQ82023', 'OKQ82024', 'OQ2023',
]
FORCE = False

_VARIANT_FLAGS = {
    'a0':    {'NLI_REORDER_ENABLED': '0', 'NLI_HINT_ENABLED': '0'},
    'a1':    {'NLI_REORDER_ENABLED': '1', 'NLI_HINT_ENABLED': '0'},
    'a2':    {'NLI_REORDER_ENABLED': '0', 'NLI_HINT_ENABLED': '1'},
    'v_new': {'NLI_REORDER_ENABLED': '1', 'NLI_HINT_ENABLED': '1'},
}

_flags = _VARIANT_FLAGS[VARIANT_ID]
print(f'Variant: {VARIANT_ID} | REORDER={_flags["NLI_REORDER_ENABLED"]} | HINT={_flags["NLI_HINT_ENABLED"]} | FORCE={FORCE} | {len(REPORT_IDS)} reports')

Variant: a0 | REORDER=0 | HINT=0 | FORCE=False | 16 reports


## Setup

In [2]:
import os
import sys
import asyncio
import shutil
import time
import traceback
from pathlib import Path
from dotenv import load_dotenv
from datetime import datetime, timezone
import pandas as pd

REPO_ROOT = Path(r'D:\Final_GRAG')
EXP_DIR = REPO_ROOT / 'experiments' / 'module3_nli'
DEST_ROOT = EXP_DIR / 'variant_runs'
LATENCY_LOG = EXP_DIR / 'data' / 'latency_per_variant_observed.csv'
LATENCY_LOG.parent.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(REPO_ROOT))

load_dotenv(REPO_ROOT / '.env')

os.environ.update(_flags)
print(f'NLI_REORDER_ENABLED={os.environ["NLI_REORDER_ENABLED"]} | NLI_HINT_ENABLED={os.environ["NLI_HINT_ENABLED"]}')

NLI_REORDER_ENABLED=0 | NLI_HINT_ENABLED=0


In [3]:
from src.compliance.graph import build_compliance_graph
from src.compliance import io as io_mod
import src.compliance.config as cfg

graph = build_compliance_graph(checkpoint=False)

## Run variants

In [4]:
def _is_done(rid):
    return (DEST_ROOT / VARIANT_ID / rid / 'compliance_report.json').exists()

done = [r for r in REPORT_IDS if _is_done(r)] if not FORCE else []
pending = [r for r in REPORT_IDS if r not in done]
print(f'{len(done)} already done | {len(pending)} pending')

16 already done | 0 pending


In [5]:
async def _run_one_report(rid):
    t0 = time.time()
    state = {
        'report_id': rid,
        'inputs': io_mod.load_report_inputs(rid),
        'phase_results': {},
        'pending_omissions': [],
    }
    final_state = await graph.ainvoke(state)
    
    elapsed = time.time() - t0
    io_mod.write_outputs(final_state)
    final_pr = final_state.get('phase_results', {}).get('final')
    overall = bool(final_pr.artifacts.get('overall_pass', False)) if final_pr else False
    return {'rid': rid, 'overall': 'PASS' if overall else 'FAIL', 'elapsed_s': round(elapsed, 1)}

In [6]:
def _copy_to_variant_runs(rid):
    src_dir = Path(cfg.REPORT_UNITS_DIR) / rid
    dst_dir = DEST_ROOT / VARIANT_ID / rid
    dst_dir.mkdir(parents=True, exist_ok=True)
    for fname in ('compliance_report.json', 'compliance_summary.csv'):
        src = src_dir / fname
        if src.exists():
            shutil.copy2(src, dst_dir / fname)

In [7]:
async def _run_all():
    out = []
    n_total = len(REPORT_IDS)
    for rank, rid in enumerate(REPORT_IDS):
        if rid in done:
            continue
        ts = datetime.now(timezone.utc).strftime('%H:%M:%S')
        print(f'[{ts}] [{rank + 1}/{n_total}] {rid}: running...')
        t0 = time.time()
        try:
            res = await _run_one_report(rid)
            t1 = time.time()
            print(f'    {res["overall"]} | {res["elapsed_s"]}s')
            _copy_to_variant_runs(rid)
            out.append(res)

            # Ghi 1 dòng log cho report vừa chạy xong; replace nếu (variant, report_id) đã tồn tại
            new_row = pd.DataFrame([{
                'variant': VARIANT_ID,
                'rank': rank,
                'report_id': rid,
                'finish_time': datetime.fromtimestamp(t1).isoformat(timespec='seconds'),
                'duration_min': round((t1 - t0) / 60, 2),
            }])
            if LATENCY_LOG.exists():
                df = pd.read_csv(LATENCY_LOG)
                df = df[~((df['variant'] == VARIANT_ID) & (df['report_id'] == rid))]
                df = pd.concat([df, new_row], ignore_index=True)
            else:
                df = new_row
            df.to_csv(LATENCY_LOG, index=False)
        except KeyboardInterrupt:
            print('KeyboardInterrupt; stopping')
            break
        except Exception as e:
            print(f'    ERROR: {e!r}')
            traceback.print_exc()
            out.append({'rid': rid, 'overall': 'ERROR', 'elapsed_s': None})
    return out

In [8]:
if pending:
    results = asyncio.run(_run_all())
    n_pass = sum(1 for r in results if r['overall'] == 'PASS')
    n_fail = sum(1 for r in results if r['overall'] == 'FAIL')
    n_err  = sum(1 for r in results if r['overall'] == 'ERROR')
    print(f'Done: {n_pass} PASS / {n_fail} FAIL / {n_err} ERROR')
else:
    print('Nothing to run.')

Nothing to run.


## Preview log

In [9]:
if LATENCY_LOG.exists():
    _log = pd.read_csv(LATENCY_LOG)
    print(_log[_log['variant'] == VARIANT_ID].to_string(index=False))
else:
    print(f'No log file at {LATENCY_LOG}')

variant  rank     report_id         finish_time  duration_min
     a0     0  Bangchak2024 2026-05-08T08:08:01           NaN
     a0     1  Bangchak2025 2026-05-08T08:58:28         50.44
     a0     2     Bapco2023 2026-05-08T09:28:59         30.52
     a0     3     Bapco2024 2026-05-08T10:01:54         32.92
     a0     4    Bunduq2023 2026-05-08T10:33:22         31.46
     a0     5     Desfa2023 2026-05-08T11:07:51         34.49
     a0     6  Energean2023 2026-05-08T11:08:43           NaN
     a0     7  Energean2024 2026-05-08T11:51:53         43.16
     a0     8      HPCL2023 2026-05-08T12:39:28         47.58
     a0     9      HPCL2024 2026-05-08T13:20:06         40.64
     a0    10 IslandOil2024 2026-05-08T13:42:51         22.76
     a0    11  Mubadala2023 2026-05-08T14:26:26         43.58
     a0    12  Mubadala2024 2026-05-08T15:06:47         40.34
     a0    13      OKQ82023 2026-05-08T15:44:59         38.20
     a0    14      OKQ82024 2026-05-08T16:22:29         37.51
     a0 